# Section II Fundamentals Evidence Lab (v3 Wrapper)

This notebook runs the full-transition Section II extraction pipeline without touching old Section II notebooks.

Pipeline script:
- `analysis/nb/section2_v3_pipeline.py`

Main output directory:
- `analysis/II_ev_v3/`

The pipeline includes:
- Input path manifest (Section II governance + COMST recipes + PRISMA + survey flow + corpus)
- Full source scan (`processed_markdowns` + `visual_analysis.txt` + `O_ISAC_*_v4.json`)
- Groq LLM-required classification (no silent baseline fallback)
- Section II-A/B/C/D/E evidence export
- Retrieval/evidence graph/cluster artifacts (`retrieval_hits.jsonl`, `evidence_graph.jsonl`, `cluster_map.csv`)
- Section II-F summary + Section II-G dual-view audit outputs
- Contract violations + readiness report


In [7]:
# @title 1. Install Dependencies
!pip install -q groq rapidfuzz tqdm


In [8]:
# @title 2. Setup & Mount Drive
from pathlib import Path
import os

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
if IN_COLAB:
    drive.mount('/content/drive')
    if os.path.exists(BASE_DIR):
        os.chdir(BASE_DIR)
        print('Working dir:', os.getcwd())
    else:
        print('Path not found:', BASE_DIR)
else:
    print('Colab not detected. Current dir:', os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [9]:
# @title 3. Validate Groq API Key (Required)
import os

api_key = os.environ.get('GROQ_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('GROQ_API_KEY is required. Add it in Colab Secrets or environment variables.')

os.environ['GROQ_API_KEY'] = api_key
print('GROQ_API_KEY loaded. Section2 v3 will run with Groq LLM classification.')


GROQ_API_KEY loaded. Section2 v3 will run with Groq LLM classification.


In [10]:
# @title 4. Preview Required Input Sets
import importlib.util
from pathlib import Path

script_path = Path('analysis/nb/section2_v3_pipeline.py')
if not script_path.exists():
    raise FileNotFoundError(f'Missing pipeline script: {script_path}')

spec = importlib.util.spec_from_file_location('section2_v3_pipeline', script_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

print('Pipeline script:', script_path)
print('Output dir:', mod.OUTPUT_DIR)
print('\nRequired input categories:')
for k, vals in mod.REQUIRED_INPUTS.items():
    print(f'- {k}: {len(vals)} paths')


Pipeline script: analysis/nb/section2_v3_pipeline.py
Output dir: analysis/II_ev_v3

Required input categories:
- section2_governance: 11 paths
- comst_recipes: 9 paths
- prisma_protocol: 8 paths
- survey_flow: 4 paths
- corpus: 3 paths


In [ ]:
# @title 5. Run Section II v3 Pipeline
!python analysis/nb/section2_v3_pipeline.py


[2026-02-10T20:23:48.922974+00:00] stage-start | stage=input_manifest_written | output_dir=analysis/II_ev_v3
[2026-02-10T20:23:55.589783+00:00] source-inventory | json_papers=221 | markdown_docs=531 | json_docs=221 | total_scan_docs=752
JSON papers: 221
Markdown docs: 531
JSON docs: 221
Total scan docs: 752
LLM classification: enabled
RUN_PROFILE=FULL_RESCAN: ignoring existing candidate file and rescanning all sources.
[2026-02-10T20:23:55.795931+00:00] full-rescan | action=ignoring_existing_candidates | path=analysis/II_ev_v3/s2_all_cand_v3.csv
[2026-02-10T20:23:55.800754+00:00] stage-start | stage=variant_generation
[2026-02-10T20:23:55.804294+00:00] variant-generation-done | concept_groups=6 | cached_terms=51 | llm_attempts=0
scan:   0% 2/752 [00:14<1:18:27,  6.28s/it][2026-02-10T20:24:11.949943+00:00] llm-progress | attempts=50 | success=50 | fail=0 | model=meta-llama/llama-4-scout-17b-16e-instruct
scan:   0% 3/752 [00:26<1:52:02,  8.97s/it][2026-02-10T20:24:58.218887+00:00] llm-pr

In [ ]:
# @title 6. Review Readiness + Violations
from pathlib import Path
import pandas as pd

out = Path('analysis/II_ev_v3')
report = out / 'readiness_report.md'
viol = out / 'contract_violations.csv'
manifest = out / 'input_manifest.md'

if manifest.exists():
    print('===== INPUT MANIFEST =====')
    print(manifest.read_text(encoding='utf-8', errors='ignore'))

if report.exists():
    print('===== READINESS REPORT =====')
    print(report.read_text(encoding='utf-8', errors='ignore'))
else:
    print('Readiness report not found:', report)

if viol.exists():
    v = pd.read_csv(viol)
    print('===== VIOLATION SUMMARY =====')
    print(v.groupby(['category', 'severity']).size().reset_index(name='count').to_string(index=False))
    print('\nSample rows:')
    print(v.head(20).to_string(index=False))
else:
    print('Violation file not found:', viol)


In [ ]:
# @title 7. Live Process Log (Runtime)
from pathlib import Path
import json

out = Path('analysis/II_ev_v3')
logp = out / 'runtime_progress.log'
snap = out / 'checkpoints' / 'runtime_progress.json'
summary = out / 'runtime_summary.json'

print('===== LAST PROGRESS LINES =====')
if logp.exists():
    lines = logp.read_text(encoding='utf-8', errors='ignore').splitlines()
    for ln in lines[-30:]:
        print(ln)
else:
    print('runtime_progress.log not found yet.')

print('\n===== CURRENT SNAPSHOT =====')
if snap.exists():
    print(json.dumps(json.loads(snap.read_text(encoding='utf-8')), indent=2))
else:
    print('runtime_progress.json not found yet.')

print('\n===== FINAL RUNTIME SUMMARY =====')
if summary.exists():
    print(json.dumps(json.loads(summary.read_text(encoding='utf-8')), indent=2))
else:
    print('runtime_summary.json not found yet (appears near end of run).')
